In [1]:

from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock
import pandas as pd
import re
import time
import nltk
import spacy
import numpy as np
from collections import Counter
from threading import local

# Load English tokenizer, POS tagger, parser, NER, and word vectors
nlp = spacy.load("en_core_web_sm")

# Define a word list using nltk corpus
english_vocab = set(w.lower() for w in nltk.corpus.words.words())

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_12"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_22"),
    os.getenv("OPENAI_KEY_23"),
    os.getenv("OPENAI_KEY_24"),
    os.getenv("OPENAI_KEY_25")
]

2023-06-14 11:28:03.868971: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
original_transcript_path = "../runs/2008-09-26-debate-preclean.txt"

In [3]:
def process_transcript(file_path, speakers_set):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Initialize an empty list to store the speakers and their speeches
    processed_transcript = []

    # Initialize an empty string to store the current speaker
    current_speaker = ''

    for line in lines:
        # Remove leading and trailing whitespaces
        line = line.strip()

        if not line:
            continue

        # Split the line into speaker and speech, the maxsplit parameter is set to 1
        split_line = line.split(':', 1)

        # Check if the line could be split into speaker and speech
        if len(split_line) == 2:
            # Potential speaker
            potential_speaker = split_line[0].strip().upper()
            
            # If the potential speaker is in our known speakers set
            if potential_speaker in speakers_set:
                # Update the current speaker and append the speech to the transcript
                current_speaker, speech = split_line
                current_speaker = current_speaker.strip().upper()
                speech = speech.strip()
                
                processed_transcript.append((current_speaker, speech))
            else:
                # If the line could not be split into a valid speaker and speech, append it to the last speaker's last speech
                processed_transcript[-1] = (processed_transcript[-1][0], processed_transcript[-1][1] + ' ' + line)
        else:
            # If the line could not be split, append it to the last speaker's last speech
            processed_transcript[-1] = (processed_transcript[-1][0], processed_transcript[-1][1] + ' ' + line)

    # Create a dictionary to count speaker turns
    speaker_turns_dict = {speaker: sum(1 for t in processed_transcript if t[0] == speaker) for speaker in speakers_set}

    return processed_transcript, speaker_turns_dict

In [4]:
def write_transcript_to_file(transcript, output_path):
    with open(output_path, 'w') as file:
        for speaker_name, speaker_content in transcript:
            # Each line starts with the speaker name followed by their speech
            file.write(f'{speaker_name}: {speaker_content}\n')


In [5]:
known_speakers = {'LEHRER', 'OBAMA', 'MCCAIN'}

In [6]:
processed_transcript, speaker_turns_dict = process_transcript(original_transcript_path, known_speakers)

output_file_name = '2008-9-29-clean-final.txt'

write_transcript_to_file(processed_transcript, output_file_name)

In [7]:
def transcript_to_dataframe(processed_transcript):
    df = pd.DataFrame(processed_transcript, columns=['speaker', 'content'])
    return df

transcript_df = transcript_to_dataframe(processed_transcript)
print(type(transcript_df))
display(transcript_df)

<class 'pandas.core.frame.DataFrame'>


,speaker,content
0,LEHRER,Good evening from the Ford Center for the Perf...
1,OBAMA,"Well, thank you very much, Jim, and thanks to ..."
2,LEHRER,"Senator McCain, two minutes."
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody...."
4,LEHRER,"All right, let’s go back to my question. How d..."
...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo..."
373,OBAMA,"Well, let me just make a closing point. You kn..."
374,LEHRER,Few seconds. We’re almost finished.
375,MCCAIN,"Jim, when I came home from prison, I saw our v..."


In [8]:
def configure_transcript_dataframe(df):
    df['turn_number'] = df.index + 1
    df['cumulative_speaker_turn_number'] = df.groupby('speaker').cumcount() + 1

    default_values = {
    'word_count_points': 0,
    'sentence_count_points': 0,
    'named_entity_count_points': 0,
    'named_entities': 'empty',
    'bonus_word_points': 0,
    'bonus_word_words': 'empty',
    'base_score': 0.0,
    'E': 0.0, # change
    'P': 0.0, # change
    'L': 0.0, # change
    'ethos_score_explained': 'empty',
    'pathos_score_explained': 'empty',
    'logos_score_explained': 'empty',
    'epl_multiplier': 0.0,
    'total_turn_score': 0.0,
    'cumulative_speaker_base_score': 0.0,
    'cumulative_speaker_total_score': 0.0,
    }

    for col, default_value in default_values.items():
        df[col] = default_value

    return df

scoreboard_df = configure_transcript_dataframe(transcript_df)
scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,0,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [9]:
def get_word_count_points(row):
    word_count = len(nltk.word_tokenize(row['content']))
    return word_count

scoreboard_df['word_count_points'] = scoreboard_df.apply(get_word_count_points, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,0,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [10]:
def get_sentence_count_points(row):
    sentence_count = len(nltk.sent_tokenize(row['content']))
    return sentence_count

scoreboard_df['sentence_count_points'] = scoreboard_df.apply(get_sentence_count_points, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [11]:
def get_named_entity_points(row):
    named_entity_count = len(nlp(row['content']).ents)
    return named_entity_count

scoreboard_df['named_entity_count_points'] = scoreboard_df.apply(get_named_entity_points, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,empty,0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [12]:
def get_named_entities(row):
    named_entities = nlp(row['content']).ents
    return named_entities

scoreboard_df['named_entities'] = scoreboard_df.apply(get_named_entities, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",0,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [13]:
def get_word_bonus_points(row):
    sentence_count = len(set(word.lower() for word in nltk.word_tokenize(row['content']) if word.lower() in english_vocab))
    return sentence_count

scoreboard_df['bonus_word_points'] = scoreboard_df.apply(get_word_bonus_points, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,empty,...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [14]:
def get_bonus_word_words(row):
    sentence_count = (set(word.lower() for word in nltk.word_tokenize(row['content']) if word.lower() in english_vocab))
    return sentence_count

scoreboard_df['bonus_word_words'] = scoreboard_df.apply(get_bonus_word_words, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,"{jim, silent, begin, will, audience, they, not...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,"{jim, sure, struggling, measured, think, will,...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,"{two, senator}",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,"{jim, lose, point, will, package, they, accoun...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,"{stand, on, my, here, about, to, go, this, rec...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,"{judgment, reform, think, they, not, to, georg...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,"{judgment, invest, point, think, to, their, fr...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,"{we, few, almost, finished, re}",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,"{on, jim, war, sad, worked, to, guarantee, pre...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [15]:
def calc_base_score(row):
    base_score = row['word_count_points'] + row['sentence_count_points'] +row['named_entity_count_points'] + row['bonus_word_points']
    return base_score

scoreboard_df['base_score'] = scoreboard_df.apply(calc_base_score, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,"{jim, silent, begin, will, audience, they, not...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,"{jim, sure, struggling, measured, think, will,...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,"{two, senator}",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,"{jim, lose, point, will, package, they, accoun...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,"{stand, on, my, here, about, to, go, this, rec...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,"{judgment, reform, think, they, not, to, georg...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,"{judgment, invest, point, think, to, their, fr...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,"{we, few, almost, finished, re}",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,"{on, jim, war, sad, worked, to, guarantee, pre...",...,0.0,0.0,0.0,empty,empty,empty,0.0,0.0,0.0,0.0


In [16]:
def prompt_epl(row):
    if not row['content']: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Evaluate the following snippet for Ethos (E), Pathos (P), and Logos (L). (Scale: 0.0 - 9.9) For non-zero score only: provide a brief explanation, otherwise leave empty string. It should be equally difficult to score 0.0 as it would be to score 9.9. [Snippet: \'{row['content']}\'] Return only a JSON object in this schema: {{E: x.x, P: y.y, L: z.z, E_Expl: string, P_Expl:, string L_Expl: string}}. Explanations must be 1 sentence long."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt

In [17]:
def calc_epl_multiplier(row):
    epl_multiplier =  1 + ((row['E'] + row['P'] + row['L']) / 30)
    return epl_multiplier

scoreboard_df['epl_multiplier'] = scoreboard_df.apply(calc_epl_multiplier, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,"{jim, silent, begin, will, audience, they, not...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,"{jim, sure, struggling, measured, think, will,...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,"{two, senator}",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,"{jim, lose, point, will, package, they, accoun...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,"{stand, on, my, here, about, to, go, this, rec...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,"{judgment, reform, think, they, not, to, georg...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,"{judgment, invest, point, think, to, their, fr...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,"{we, few, almost, finished, re}",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,"{on, jim, war, sad, worked, to, guarantee, pre...",...,0.0,0.0,0.0,empty,empty,empty,1.0,0.0,0.0,0.0


In [18]:
print(scoreboard_df.columns)

Index(['speaker', 'content', 'turn_number', 'cumulative_speaker_turn_number',
       'word_count_points', 'sentence_count_points',
       'named_entity_count_points', 'named_entities', 'bonus_word_points',
       'bonus_word_words', 'base_score', 'E', 'P', 'L',
       'ethos_score_explained', 'pathos_score_explained',
       'logos_score_explained', 'epl_multiplier', 'total_turn_score',
       'cumulative_speaker_base_score', 'cumulative_speaker_total_score'],
      dtype='object')


In [19]:
def calc_total_turn_score(row):
    total_turn_score = row['base_score'] * row['epl_multiplier']
    return total_turn_score

scoreboard_df['total_turn_score'] = scoreboard_df.apply(calc_total_turn_score, axis=1)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,"{jim, silent, begin, will, audience, they, not...",...,0.0,0.0,0.0,empty,empty,empty,1.0,469.0,0.0,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,"{jim, sure, struggling, measured, think, will,...",...,0.0,0.0,0.0,empty,empty,empty,1.0,690.0,0.0,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,"{two, senator}",...,0.0,0.0,0.0,empty,empty,empty,1.0,11.0,0.0,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,"{jim, lose, point, will, package, they, accoun...",...,0.0,0.0,0.0,empty,empty,empty,1.0,676.0,0.0,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,"{stand, on, my, here, about, to, go, this, rec...",...,0.0,0.0,0.0,empty,empty,empty,1.0,130.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,"{judgment, reform, think, they, not, to, georg...",...,0.0,0.0,0.0,empty,empty,empty,1.0,446.0,0.0,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,"{judgment, invest, point, think, to, their, fr...",...,0.0,0.0,0.0,empty,empty,empty,1.0,331.0,0.0,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,"{we, few, almost, finished, re}",...,0.0,0.0,0.0,empty,empty,empty,1.0,16.0,0.0,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,"{on, jim, war, sad, worked, to, guarantee, pre...",...,0.0,0.0,0.0,empty,empty,empty,1.0,169.0,0.0,0.0


In [20]:
def calc_cumulative_speaker_base_score(df):
    df['cumulative_speaker_base_score'] = df.groupby('speaker')['base_score'].cumsum()
    return df

scoreboard_df = calc_cumulative_speaker_base_score(scoreboard_df)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,"{jim, silent, begin, will, audience, they, not...",...,0.0,0.0,0.0,empty,empty,empty,1.0,469.0,469,0.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,"{jim, sure, struggling, measured, think, will,...",...,0.0,0.0,0.0,empty,empty,empty,1.0,690.0,690,0.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,"{two, senator}",...,0.0,0.0,0.0,empty,empty,empty,1.0,11.0,480,0.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,"{jim, lose, point, will, package, they, accoun...",...,0.0,0.0,0.0,empty,empty,empty,1.0,676.0,676,0.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,"{stand, on, my, here, about, to, go, this, rec...",...,0.0,0.0,0.0,empty,empty,empty,1.0,130.0,610,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,"{judgment, reform, think, they, not, to, georg...",...,0.0,0.0,0.0,empty,empty,empty,1.0,446.0,25679,0.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,"{judgment, invest, point, think, to, their, fr...",...,0.0,0.0,0.0,empty,empty,empty,1.0,331.0,27444,0.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,"{we, few, almost, finished, re}",...,0.0,0.0,0.0,empty,empty,empty,1.0,16.0,5607,0.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,"{on, jim, war, sad, worked, to, guarantee, pre...",...,0.0,0.0,0.0,empty,empty,empty,1.0,169.0,25848,0.0


In [21]:
def calc_cumulative_speaker_total_score(df):
    df['cumulative_speaker_total_score'] = df.groupby('speaker')['total_turn_score'].cumsum()
    return df

scoreboard_df = calc_cumulative_speaker_total_score(scoreboard_df)

scoreboard_df

,speaker,content,turn_number,cumulative_speaker_turn_number,word_count_points,sentence_count_points,named_entity_count_points,named_entities,bonus_word_points,bonus_word_words,...,E,P,L,ethos_score_explained,pathos_score_explained,logos_score_explained,epl_multiplier,total_turn_score,cumulative_speaker_base_score,cumulative_speaker_total_score
0,LEHRER,Good evening from the Ford Center for the Perf...,1,1,291,16,30,"((evening), (the, Ford, Center, for, the, Perf...",132,"{jim, silent, begin, will, audience, they, not...",...,0.0,0.0,0.0,empty,empty,empty,1.0,469.0,469,469.0
1,OBAMA,"Well, thank you very much, Jim, and thanks to ...",2,1,484,19,14,"((Jim), (the, University, of, Mississippi), (t...",173,"{jim, sure, struggling, measured, think, will,...",...,0.0,0.0,0.0,empty,empty,empty,1.0,690.0,690,690.0
2,LEHRER,"Senator McCain, two minutes.",3,2,6,1,2,"((McCain), (two, minutes))",2,"{two, senator}",...,0.0,0.0,0.0,empty,empty,empty,1.0,11.0,480,480.0
3,MCCAIN,"Well, thank you, Jim. And thanks to everybody....",4,1,472,28,27,"((Jim), (tonight), (Kennedy), (Senate), (the, ...",149,"{jim, lose, point, will, package, they, accoun...",...,0.0,0.0,0.0,empty,empty,empty,1.0,676.0,676,676.0
4,LEHRER,"All right, let’s go back to my question. How d...",5,3,79,7,4,"((’s), (five, minutes), (Obama), (McCain))",40,"{stand, on, my, here, about, to, go, this, rec...",...,0.0,0.0,0.0,empty,empty,empty,1.0,130.0,610,610.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372,MCCAIN,"I’ve been involved, as I mentioned to you befo...",373,125,312,15,6,"((20, -, some, years), (Obama), (Russian), (Ge...",113,"{judgment, reform, think, they, not, to, georg...",...,0.0,0.0,0.0,empty,empty,empty,1.0,446.0,25679,25679.0
373,OBAMA,"Well, let me just make a closing point. You kn...",374,126,220,8,7,"((Kenya), (the, United, States), (Earth), (the...",96,"{judgment, invest, point, think, to, their, fr...",...,0.0,0.0,0.0,empty,empty,empty,1.0,331.0,27444,27444.0
374,LEHRER,Few seconds. We’re almost finished.,375,124,9,2,0,(),5,"{we, few, almost, finished, re}",...,0.0,0.0,0.0,empty,empty,empty,1.0,16.0,5607,5607.0
375,MCCAIN,"Jim, when I came home from prison, I saw our v...",376,126,107,3,4,"((Jim), (MIA), (two), (the, United, States))",55,"{on, jim, war, sad, worked, to, guarantee, pre...",...,0.0,0.0,0.0,empty,empty,empty,1.0,169.0,25848,25848.0


In [22]:
def recalculate_all_scores(df):
    scoreboard_df = df
    scoreboard_df['word_count_points'] = scoreboard_df.apply(get_word_count_points, axis=1)
    scoreboard_df['sentence_count_points'] = scoreboard_df.apply(get_sentence_count_points, axis=1)
    scoreboard_df['named_entity_count_points'] = scoreboard_df.apply(get_named_entity_points, axis=1)
    scoreboard_df['named_entities'] = scoreboard_df.apply(get_named_entities, axis=1)
    scoreboard_df['bonus_word_points'] = scoreboard_df.apply(get_word_bonus_points, axis=1)
    scoreboard_df['bonus_word_words'] = scoreboard_df.apply(get_bonus_word_words, axis=1)
    scoreboard_df['base_score'] = scoreboard_df.apply(calc_base_score, axis=1)
    scoreboard_df['epl_multiplier'] = scoreboard_df.apply(calc_epl_multiplier, axis=1)
    scoreboard_df['total_turn_score'] = scoreboard_df.apply(calc_total_turn_score, axis=1)    
    return df


In [23]:
def parse_content(content):
    # Parse JSON string into a dictionary
    content_dict = json.loads(content)

    # Extract all relevant columns
    relevant_columns = {key: value for key, value in content_dict.items() if key in ['E', 'P', 'L', 'E_Expl', 'P_Expl', 'L_Expl']}

    return relevant_columns

def get_gpt_response(row, api_key):
    openai.api_key = api_key
    response = prompt_epl(row)
    return response


def write_responses_to_file(response, lock1, output_file_full, idx):
    with lock1:
        with open(output_file_full, 'a') as f:
            # Append the index to the response data
            output_data = {
                'index': idx,
                'response': response
            }
            json.dump(output_data, f, indent=4)
            f.write('\n')


def recalculate_scores(df, idx):
    df.loc[idx, 'epl_multiplier'] = 1 + ((float(df.loc[idx, 'E']) + float(df.loc[idx, 'P']) + float(df.loc[idx, 'L'])) / 30)
    df.loc[idx, 'base_score'] = float(df.loc[idx, 'base_score']) 
    df.loc[idx, 'total_turn_score'] = df.loc[idx, 'base_score'] * (1 + (df.loc[idx, 'epl_multiplier']))

    df.loc[idx, 'cumulative_speaker_base_score'] = df[df['speaker'] == df.loc[idx, 'speaker']]['base_score'].sum()
    df.loc[idx, 'cumulative_speaker_total_score'] = df[df['speaker'] == df.loc[idx, 'speaker']]['total_turn_score'].sum()

def extract_and_parse_content(response):
    # Extract 'content' field and parse it to get scores and explanations
    content = response['choices'][0]['message']['content']
    parsed_content = parse_content(content)
    return parsed_content

def update_dataframe_with_parsed_content(df, parsed_content, idx):
    for key, value in parsed_content.items():
        df.loc[idx, key] = value


In [24]:
def worker(idx, output_file_full, df, lock1, lock2, api_key):
    retries = 5
    for attempt in range(retries):
        try:
            # Get current row
            row = df.loc[idx]

            # Set the API key
            openai.api_key = api_key

            # Call OpenAI API and get response
            response = prompt_epl(row)

            # Extract and parse content
            parsed_content = extract_and_parse_content(response)
            
            # Save the response to file
            write_responses_to_file(response, lock1, output_file_full, idx)

            break

        except Exception as e:
            # If it's the last attempt and it still fails, we print the error and move on to the next task
            if attempt == retries - 1:
                print(f'Error at index {idx} after {retries} attempts: {str(e)}')
                print(f'Response content: {response["choices"][0]["message"]["content"]}')  # Add this line
            else:
                # If it's not the last attempt, we wait for a bit and try again
                time.sleep(2 ** attempt)

    return idx, parsed_content


In [25]:
def get_epl(input_dataframe, output_file_full, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startturn is not None and endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with 'startturn' and 'endturn'. Use 'runcount' alone or with either 'startturn' and 'endturn'.")

    # Determine start and end points for analysis
    start = startturn if startturn is not None else 0
    end = endturn if endturn is not None else len(input_dataframe)

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(input_dataframe))

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(input_dataframe):
        end = len(input_dataframe)
        print(f"Warning: The end point exceeded the length of the dataframe. It has been adjusted to the last turn: {end}")

    # Create locks for writing to file and dataframe
    lock1 = Lock()
    lock2 = Lock()

    with ThreadPoolExecutor(max_workers=25) as executor:
        # Assign API keys cyclically and start all the tasks
        futures = [executor.submit(worker, i, output_file_full, input_dataframe, lock1, lock2, open_ai_keys[i % len(open_ai_keys)]) for i in range(start, end)]

        # Ensure all threads have finished
        for future in as_completed(futures):
            try:
                # Get the original index from the future
                original_index, parsed_content = future.result()

                # Update DataFrame with new values (with thread lock to avoid race conditions)
                with lock2:
                    update_dataframe_with_parsed_content(input_dataframe, parsed_content, original_index)
            except Exception as e:
                print(f"Error in a thread: {str(e)}")

    # Calculate the cumulative scores after all threads have completed
    for idx in input_dataframe.index:
        recalculate_scores(input_dataframe, idx)

    # Sort DataFrame by 'Turn' after all threads are completed
    input_dataframe.sort_values('turn_number', inplace=True)
    input_dataframe.reset_index(drop=True, inplace=True)

    # Reset openai.api_key to None after the job is done
    openai.api_key = None

    return input_dataframe


In [26]:
final_scoreboard = get_epl(scoreboard_df, "2008-sep-29-detailed-scoreboard-2.json", startturn=0, endturn=10)

In [ ]:
final_scoreboard = recalculate_all_scores(final_scoreboard)
final_scoreboard = calc_cumulative_speaker_base_score(final_scoreboard)
final_scoreboard = calc_cumulative_speaker_total_score(final_scoreboard)

final_scoreboard

In [34]:
final_scoreboard.to_csv("sep-29-2008-final-scoreboard-csv.csv")

In [35]:
csv_test = pd.read_csv("sep-29-2008-final-scoreboard-csv.csv")

In [ ]:
csv_test